<a href="https://colab.research.google.com/github/maclandrol/cours-ia-med/blob/master/05_Analyse_Radiographies_Thoraciques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05. Analyse de Radiographies Thoraciques avec l'IA

**Enseignant:** Emmanuel Noutahi, PhD

---

## Objectifs

Dans ce cours pratique, vous allez:
- Charger et visualiser des radiographies thoraciques
- Utiliser un modèle d'IA pré-entraîné pour détecter des pathologies
- Interpréter les résultats de prédiction
- Comprendre les limites et applications cliniques

## Contexte Médical

Les radiographies thoraciques sont l'examen d'imagerie le plus courant en médecine. L'IA peut aider à:
- Détecter des anomalies (pneumonie, cardiomégalie, épanchements, etc.)
- Prioriser les cas urgents
- Assister le radiologue (mais ne le remplace pas!)

**Important:** L'IA est un outil d'aide au diagnostic. La décision finale revient toujours au médecin.

## 1. Installation et Configuration

Nous utilisons **TorchXRayVision**, une bibliothèque spécialisée pour l'analyse de radiographies.

In [ ]:
# Installation des bibliothèques nécessaires
!pip install -q torchxrayvision torch torchvision matplotlib numpy pillow

print("Installation terminée!")

In [ ]:
# Importation des bibliothèques
import torch
import torchxrayvision as xrv
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# Configuration de l'affichage
plt.rcParams['figure.figsize'] = (12, 8)

# Vérification du GPU (optionnel, fonctionne aussi sur CPU)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositif utilisé: {device}")
print(f"Version PyTorch: {torch.__version__}")
print(f"Version TorchXRayVision: {xrv.__version__}")

## 2. Chargement du Modèle d'IA

Nous utilisons un modèle **DenseNet121** pré-entraîné sur des millions de radiographies.
Ce modèle peut détecter 18 pathologies différentes.

In [ ]:
# Chargement du modèle pré-entraîné
print("Chargement du modèle d'IA...")
model = xrv.models.DenseNet(weights="densenet121-res224-all")
model.to(device)
model.eval()  # Mode évaluation (pas d'entraînement)

print("Modèle chargé avec succès!")
print(f"\nPathologies détectables ({len(model.pathologies)}):")
for i, pathology in enumerate(model.pathologies, 1):
    print(f"  {i:2d}. {pathology}")

## 3. Téléchargement d'une Radiographie d'Exemple

Nous allons utiliser une image d'exemple du dataset NIH.

In [ ]:
# Téléchargement d'une image d'exemple
!wget -q https://github.com/ieee8023/covid-chestxray-dataset/raw/master/images/01E392EE-69F9-4E33-BFCE-E5C968654078.jpeg -O example_xray.jpg

# Chargement de l'image
img_path = "example_xray.jpg"
img = Image.open(img_path)

# Affichage de l'image originale
plt.figure(figsize=(8, 8))
plt.imshow(img, cmap='gray')
plt.title("Radiographie Thoracique - Image Originale", fontsize=14, fontweight='bold')
plt.axis('off')
plt.show()

print(f"Dimensions de l'image: {img.size}")

## 4. Préparation de l'Image pour le Modèle

Le modèle d'IA nécessite que l'image soit préparée dans un format spécifique:
- Convertie en niveaux de gris
- Redimensionnée à 224x224 pixels
- Valeurs pixel dans [0, 255] (le modèle applique sa propre normalisation interne)

In [ ]:
def preparer_image_pour_modele(img_path):
    """
    Prépare une radiographie pour l'analyse par le modèle d'IA.
    
    Étapes:
    1. Charger l'image
    2. Convertir en niveaux de gris
    3. Redimensionner à 224x224
    4. Normaliser les valeurs en [0, 255]
    5. Convertir en tensor PyTorch
    
    Note: TorchXRayVision attend des valeurs pixel dans [0, 255] ou [-1024, 1024]
    Le modèle applique sa propre normalisation interne.
    """
    # 1. Charger l'image
    img = Image.open(img_path)
    
    # 2. Convertir en niveaux de gris
    if img.mode != 'L':
        img = img.convert('L')
    
    # 3. Redimensionner à 224x224
    img = img.resize((224, 224))
    
    # 4. Convertir en array numpy
    img_array = np.array(img, dtype=np.float32)
    
    # 5. S'assurer que les valeurs sont dans [0, 255]
    # (normalement déjà le cas pour des images JPEG/PNG)
    if img_array.max() <= 1.0:
        img_array = img_array * 255.0
    
    # 6. Convertir en tensor PyTorch
    # Format attendu: [batch, canaux, hauteur, largeur] = [1, 1, 224, 224]
    img_tensor = torch.FloatTensor(img_array)
    img_tensor = img_tensor.unsqueeze(0).unsqueeze(0)  # Ajouter dimensions batch et canal
    img_tensor = img_tensor.to(device)
    
    return img_tensor, img_array

# Préparation de notre image
img_tensor, img_processed = preparer_image_pour_modele(img_path)

# Affichage de l'image traitée
plt.figure(figsize=(6, 6))
plt.imshow(img_processed, cmap='gray')
plt.title("Image Préparée pour le Modèle (224x224)", fontsize=12, fontweight='bold')
plt.axis('off')
plt.show()

print(f"Forme du tensor: {img_tensor.shape}")
print(f"Plage de valeurs: [{img_tensor.min():.2f}, {img_tensor.max():.2f}]")
print("L'image est maintenant prête pour l'analyse!")

## 5. Analyse de la Radiographie

Maintenant, nous utilisons le modèle pour prédire la présence de pathologies.

In [ ]:
# Prédiction avec le modèle
print("Analyse en cours...")

with torch.no_grad():  # Pas besoin de calculer les gradients (mode évaluation)
    predictions = model(img_tensor)
    # Convertir les sorties en probabilités (entre 0 et 1)
    probabilities = torch.sigmoid(predictions).cpu().numpy()[0]

print("Analyse terminée!\n")

# Affichage des résultats
print("=" * 60)
print("RÉSULTATS DE L'ANALYSE")
print("=" * 60)

# Créer un dictionnaire pathologie -> probabilité
resultats = {pathology: prob for pathology, prob in zip(model.pathologies, probabilities)}

# Trier par probabilité décroissante
resultats_tries = sorted(resultats.items(), key=lambda x: x[1], reverse=True)

# Afficher toutes les pathologies
for i, (pathology, prob) in enumerate(resultats_tries, 1):
    # Indicateur visuel
    if prob > 0.5:
        indicateur = "[DÉTECTÉ]"
        couleur = "rouge"
    elif prob > 0.3:
        indicateur = "[SUSPECT]"
        couleur = "orange"
    else:
        indicateur = "          "
        couleur = "vert"
    
    print(f"{i:2d}. {pathology:25s} {prob*100:5.1f}% {indicateur}")

print("=" * 60)

## 6. Visualisation des Résultats

Créons un graphique pour mieux comprendre les résultats.

In [ ]:
# Création d'un graphique à barres
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Graphique 1: Image + Top 5 détections
ax1.imshow(img, cmap='gray')
ax1.set_title("Radiographie Analysée", fontsize=14, fontweight='bold')
ax1.axis('off')

# Graphique 2: Barres des probabilités
pathologies_list = [p for p, _ in resultats_tries]
probs_list = [prob * 100 for _, prob in resultats_tries]

# Couleurs selon le niveau de probabilité
couleurs = ['red' if p > 50 else 'orange' if p > 30 else 'lightblue' for p in probs_list]

bars = ax2.barh(range(len(pathologies_list)), probs_list, color=couleurs, alpha=0.7)
ax2.set_yticks(range(len(pathologies_list)))
ax2.set_yticklabels(pathologies_list, fontsize=9)
ax2.set_xlabel('Probabilité (%)', fontsize=12)
ax2.set_title('Probabilités de Détection par Pathologie', fontsize=14, fontweight='bold')
ax2.axvline(x=50, color='red', linestyle='--', linewidth=1, label='Seuil de détection (50%)')
ax2.axvline(x=30, color='orange', linestyle='--', linewidth=1, label='Seuil de suspicion (30%)')
ax2.legend()
ax2.grid(axis='x', alpha=0.3)
ax2.set_xlim(0, 100)

plt.tight_layout()
plt.show()

## 7. Interprétation Clinique

Comment interpréter ces résultats en tant que professionnel de santé?

In [ ]:
def generer_interpretation(resultats_tries, seuil_haut=0.5, seuil_bas=0.3):
    """
    Génère une interprétation clinique simple des résultats.
    """
    detections_hautes = [(p, prob) for p, prob in resultats_tries if prob > seuil_haut]
    detections_moderees = [(p, prob) for p, prob in resultats_tries if seuil_bas < prob <= seuil_haut]
    
    print("\n" + "="*70)
    print("INTERPRÉTATION CLINIQUE ASSISTÉE PAR IA")
    print("="*70)
    
    if detections_hautes:
        print("\n ANOMALIES PROBABLES (probabilité > 50%):")
        for pathology, prob in detections_hautes:
            print(f"   - {pathology}: {prob*100:.1f}%")
        print("   --> Recommandation: Revue radiologique prioritaire")
    
    if detections_moderees:
        print("\n ANOMALIES À SURVEILLER (probabilité 30-50%):")
        for pathology, prob in detections_moderees:
            print(f"   - {pathology}: {prob*100:.1f}%")
        print("   --> Recommandation: Corrélation clinique recommandée")
    
    if not detections_hautes and not detections_moderees:
        print("\n ANALYSE:")
        print("   Aucune anomalie majeure détectée par l'IA.")
        print("   --> Recommandation: Revue de routine par radiologue")
    
    print("\n" + "="*70)
    print("RAPPELS IMPORTANTS:")
    print("="*70)
    print(" 1. L'IA est un OUTIL D'AIDE, pas un diagnostic définitif")
    print(" 2. Une revue par un radiologue qualifié est TOUJOURS nécessaire")
    print(" 3. La décision clinique doit intégrer: examen, anamnèse, bio, imagerie")
    print(" 4. Les faux positifs et faux négatifs sont possibles")
    print(" 5. L'IA aide à prioriser et à ne pas manquer d'anomalies")
    print("="*70 + "\n")

# Génération de l'interprétation
generer_interpretation(resultats_tries)

## 8. Exercice Pratique: Analyser Votre Propre Image

Téléchargez une radiographie thoracique et analysez-la avec le modèle.

In [ ]:
# Option 1: Télécharger depuis une URL
# Décommentez et modifiez l'URL ci-dessous
# !wget -q <VOTRE_URL_ICI> -O ma_radio.jpg
# mon_image = "ma_radio.jpg"

# Option 2: Upload depuis Google Colab
from google.colab import files
uploaded = files.upload()
if uploaded:
    mon_image = list(uploaded.keys())[0]
    print(f"Image téléchargée: {mon_image}")
    
    # Analyse de votre image
    img_tensor, img_processed = preparer_image_pour_modele(mon_image)
    
    with torch.no_grad():
        predictions = model(img_tensor)
        probabilities = torch.sigmoid(predictions).cpu().numpy()[0]
    
    resultats = {pathology: prob for pathology, prob in zip(model.pathologies, probabilities)}
    resultats_tries = sorted(resultats.items(), key=lambda x: x[1], reverse=True)
    
    # Affichage
    img_display = Image.open(mon_image)
    plt.figure(figsize=(8, 8))
    plt.imshow(img_display, cmap='gray')
    plt.title("Votre Radiographie", fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.show()
    
    generer_interpretation(resultats_tries)
else:
    print("Aucune image téléchargée.")

## 9. Résumé et Points Clés

### Ce que vous avez appris:

1. **Chargement de modèles pré-entraînés**: Utiliser des modèles d'IA sans avoir à les créer
2. **Préparation d'images médicales**: Normalisation et formatage pour l'IA
3. **Interprétation de résultats**: Comprendre les probabilités et seuils de décision
4. **Limites de l'IA**: L'IA aide mais ne remplace pas le jugement clinique

### Applications pratiques:

- **Triage automatique**: Prioriser les cas urgents dans un service d'urgence
- **Double lecture**: Assister les radiologues pour éviter les oublis
- **Télémédecine**: Pré-analyse dans les zones avec peu de radiologues
- **Recherche**: Analyse de grandes cohortes de patients

### Limites importantes:

1. **Qualité de l'image**: Le modèle est sensible à la qualité de la radiographie
2. **Biais de données**: Entraîné sur certaines populations, peut être moins performant sur d'autres
3. **Pathologies rares**: Moins fiable pour des conditions peu représentées dans les données d'entraînement
4. **Contexte clinique**: Ne connaît pas l'histoire du patient, les symptômes, etc.

### Prochaines étapes:

- Cours 6: Segmentation d'images médicales (délimiter les organes/lésions)
- Cours 7: Éthique et déploiement responsable de l'IA en médecine

## 10. Questions de Réflexion

1. **Seuils de décision**: Pourquoi avons-nous choisi 50% comme seuil de détection? Que se passe-t-il si on le baisse à 30% ou le monte à 70%?

2. **Faux positifs vs faux négatifs**: Dans le contexte médical, qu'est-ce qui est plus grave? Un faux positif (alarme injustifiée) ou un faux négatif (pathologie manquée)?

3. **Responsabilité**: Si l'IA manque une pathologie qu'un radiologue aurait détectée, qui est responsable?

4. **Généralisation**: Ce modèle a été entraîné sur des données américaines. Sera-t-il aussi performant sur des patients africains, asiatiques, pédiatriques?

5. **Utilité clinique**: Dans quelles situations cette IA serait-elle la plus utile dans votre pratique?